[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/solutions/86_fly_catching_pdd260702_solution.ipynb)

# Solution: Catch the Fly

Reference solution.

## 解析

**结论：功能图（每点出度 1）里，任何苍蝇最终都会陷入某个环。要保证在任意起点都能被抓，等价于「每个环上至少放一个陷阱」。给非环点放陷阱是浪费。最小花费 = 各环上最小 `c` 之和。**

### 为什么只看环
出度恒为 1，从任意点沿 `a` 走下去必然进入一个环并永远绕圈。只要环上有一个被陷阱覆盖的房间，绕圈的苍蝇迟早经过它被抓；反之若某环上无陷阱，起于该环的苍蝇永远抓不到。环外（“尾巴”上的）房间放陷阱不影响可达性，纯属浪费。

### 找环：拓扑剥离
统计入度，反复移除入度为 0 的点（它们不可能在环上），并把其后继入度减 1。剩下没被移除的点就是所有环的并集。

### 取每环最小值
对每个未访问的存活点，沿 `a` 走一圈回到自身，途中取 `c` 的最小值累加。

### 验证
已用「显式判定每个点是否在环上」的暴力在数千组随机功能图上对拍一致。

### 复杂度
拓扑剥离 `O(n)`，遍历环 `O(n)`，总 `O(n)` 时间、`O(n)` 空间。

In [ ]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass

In [ ]:
from typing import List
from collections import deque

In [ ]:
# ✅ SOLUTION

class Solution:
    def min_cost(self, a: List[int], c: List[int]) -> int:
        n = len(a)
        indeg = [0] * n
        for i in range(n):
            indeg[a[i]] += 1
        removed = [False] * n
        q = deque(i for i in range(n) if indeg[i] == 0)
        while q:                                   # topological peel of the 'tails'
            u = q.popleft(); removed[u] = True
            v = a[u]; indeg[v] -= 1
            if indeg[v] == 0 and not removed[v]:
                q.append(v)
        visited = [False] * n; ans = 0
        for i in range(n):                         # survivors form the cycles
            if not removed[i] and not visited[i]:
                mn = c[i]; visited[i] = True; x = a[i]
                while x != i:
                    visited[x] = True; mn = min(mn, c[x]); x = a[x]
                ans += mn                          # cheapest trap on this cycle
        return ans

In [ ]:
# Demo
sol = Solution()
print(sol.min_cost([1, 3, 1, 1], [1, 10, 2, 10]))   # 10
print(sol.min_cost([0], [5]))                       # 5
print(sol.min_cost([1, 0], [4, 9]))                 # 4

In [ ]:
from torch_judge import check
check('fly_catching_pdd260702')